전처리까지 진행된 결과를 대상으로 VitalDB, MOVER의 환자들의 특징을 보고한다.

In [1]:
import os
from pathlib import Path
import h5py
import numpy as np
import pandas as pd
from tqdm import tqdm
from src.module.utils import load_config, load_npy

In [ ]:
"""
Thoracic: 흉부, 폐, 복부 등의 수술
Cardiac: 심장
Gynecologic: 부인과
Obstetric: 산부인과

VitalDB는 Cardiac & Obstetric surgery는 애초에 배제하고 수집되었으며
MOVER는 ICD-10-PCS 코드 중 앞 두자리가 다음과 같은 경우를 제외하면 되겠다:
- Cardiac = '02*'
- Obstetric = '10*'

[최종] Cardiac & Gynecologic surgery 배제
VitalDB = Exclude cardiac, gynecologic
MOVER = Exclude cardiac('02'), gynecologic('10', '0U')
"""


def vitaldb_prepare_cohort(file_path) -> pd.DataFrame:
    cohort = pd.read_csv(file_path / "clinical_data.csv").reset_index(drop=True)
    cohort["age_num"] = np.where((cohort["age"] == ">89"), 90, cohort["age"]).astype(
        np.float32  # age < 1
    )
    cohort = (
        cohort.query("18 <= age_num")
        .query("department not in ['Gynecology']")
        # .query("department not in ['Thoracic surgery', 'Gynecology']")
        .query("ane_type == 'General'")
    )
    return cohort


def mover_prepare_cohort(file_path) -> pd.DataFrame:
    pat_info = pd.read_csv(file_path / "patient_information.csv")
    pat_code = pd.read_csv(file_path / "patient_coding.csv")
    pid_to_mrn = pd.read_csv(file_path / "EPIC_MRN_PAT_ID.csv")
    selected_mrn = pat_code[
        ~pat_code.REF_BILL_CODE.apply(lambda x: str(x)[:2] in ["02", "10", "0U"])
    ]["MRN"].unique()
    cohort = (
        pd.merge(pid_to_mrn, pat_info, how="inner", on=["LOG_ID", "MRN"])
        .drop_duplicates()
        .reset_index(drop=True)
    )
    cohort = (
        cohort.query("18 <= BIRTH_DATE")
        .query("MRN.isin(@selected_mrn)")  # cardiac & obstetric surgery excluded
        .query("PRIMARY_ANES_TYPE_NM == 'General'")
        .reset_index(drop=True)
    )
    return cohort

In [3]:
def load_h5(file_path) -> tuple:
    with h5py.File(file_path, "r") as f:
        ts = f["signal/ts"][:]
        sig = f["signal/abp"][:]
        mbp = f["signal/mbp"][:]
        sqi = f["signal/sqi"][:]
        fs = f["signal"].attrs["fs"]
        sig_len = f["signal"].attrs["len"]
        cid = f.attrs["caseid"].astype(str)

    return cid, fs, sig_len, ts, sig, mbp, sqi

In [4]:
data_dir = Path("../../data")
vitaldb_cohort = vitaldb_prepare_cohort(data_dir / "vitaldb")
mover_cohort = mover_prepare_cohort(data_dir / "mover")

In [5]:
# # 전처리된 데이터만 baseline characterisitcs를 추출한다.
# vitaldb_caselist = sorted(
#     [
#         int(x.parent.name)
#         for x in (data_dir / "vitaldb" / "02_04.processed").rglob("*.h5")
#     ]
# )
# mover_caselist = sorted(
#     [x.parent.name for x in (data_dir / "mover" / "02.processed").rglob("*.h5")]
# )
# print(len(vitaldb_caselist), len(mover_caselist))

# # 주의!
# # vitaldb에서 수집된 caselist는 환자 단위가 아니다.
# # mover에서 수집된 caselist는 환자 단위이다.

In [6]:
config = load_config("../../config/conf.yaml")
src_path = Path(config["data_path"]).expanduser()

In [7]:
# vitaldb case no
vdb_pos_cid = np.hstack(
    [
        load_npy(fp)
        for fp in (src_path / "vitaldb" / "05.dataset_new").rglob(
            "hypophetversion2*pos_cid.npy"
        )
    ]
)
vdb_neg_cid = np.hstack(
    [
        load_npy(fp)
        for fp in (src_path / "vitaldb" / "05.dataset_new").rglob(
            "hypophetversion2*neg_cid.npy"
        )
    ]
)
vitaldb_caselist = np.unique(np.hstack([vdb_pos_cid, vdb_neg_cid])).astype(int)
print(f"case no: {vitaldb_caselist.size}")

case no: 3239


In [8]:
# mover case no
mov_pos_cid = np.hstack(
    [
        load_npy(fp)
        for fp in (src_path / "mover" / "05.dataset_new").rglob(
            "hypophetversion2*pos_cid.npy"
        )
    ]
)
mov_neg_cid = np.hstack(
    [
        load_npy(fp)
        for fp in (src_path / "mover" / "05.dataset_new").rglob(
            "hypophetversion2*neg_cid.npy"
        )
    ]
)
mover_caselist = np.unique(np.hstack([mov_pos_cid, mov_neg_cid]))
print(f"case no: {mover_caselist.size}")

case no: 649


In [9]:
vitaldb_cohort = vitaldb_cohort.query("caseid.isin(@vitaldb_caselist)").reset_index(
    drop=True
)
print("VitalDB case no: ", vitaldb_cohort.caseid.nunique())
print("VitalDB patient no: ", vitaldb_cohort.subjectid.nunique())

mover_cohort = mover_cohort.query("PAT_ID.isin(@mover_caselist)").reset_index(drop=True)
print("MOVER case no: ", mover_cohort.LOG_ID.nunique())
print("MOVER patient no: ", mover_cohort.PAT_ID.nunique())

VitalDB case no:  3239
VitalDB patient no:  3118
MOVER case no:  1199
MOVER patient no:  649


# Age

In [10]:
print("VitalDB: ", vitaldb_cohort.age_num.mean(), "±", vitaldb_cohort.age_num.std())
print("MOVER: ", mover_cohort.BIRTH_DATE.mean(), "±", mover_cohort.BIRTH_DATE.std())

VitalDB:  59.442112 ± 14.205725
MOVER:  60.737281067556296 ± 15.678857757540955


# Sex

In [11]:
print(
    "VitalDB: ",
    sum(vitaldb_cohort.sex == "M"),
    sum(vitaldb_cohort.sex == "M") / len(vitaldb_cohort),
)
print(
    "MOVER: ",
    sum(mover_cohort.SEX == "Male"),
    sum(mover_cohort.SEX == "Male") / len(mover_cohort),
)

VitalDB:  1917 0.5918493362148811
MOVER:  661 0.5512927439532944


# Height

In [12]:
print("VitalDB: ", vitaldb_cohort.height.mean(), "±", vitaldb_cohort.height.std())
# print("MOVER: ", mover_cohort.HEIGHT.astype().mean(), "±", mover_cohort.HEIGHT.std())
print("MOVER: 168.70 ± 10.90")  # Claude (inch to cm)

VitalDB:  163.12380364309973 ± 8.681527013404438
MOVER: 168.70 ± 10.90


# Weight

In [13]:
print("VitalDB: ", vitaldb_cohort.weight.mean(), "±", vitaldb_cohort.weight.std())
# print("MOVER: ", mover_cohort.WEIGHT.mean(), "±", mover_cohort.WEIGHT.std())
print("MOVER: 78.30 ± 20.30")  # Claude (oz to kg)

VitalDB:  61.41239271380055 ± 11.466824119656087
MOVER: 78.30 ± 20.30


# Surgery type

In [14]:
display(
    vitaldb_cohort.groupby("optype")
    .count()[["caseid"]]
    .reset_index()
    .sort_values("caseid", ascending=False)
)
print(
    vitaldb_cohort.groupby("optype")
    .count()[["caseid"]]
    .reset_index()
    .sort_values("caseid", ascending=False)
    .sum()
)

,optype,caseid
6,Others,531
5,Minor resection,457
7,Stomach,438
4,Major resection,437
2,Colorectal,341
0,Biliary/Pancreas,318
9,Transplantation,285
3,Hepatic,254
10,Vascular,87
8,Thyroid,77


optype    OthersMinor resectionStomachMajor resectionCol...
caseid                                                 3239
dtype: object


In [15]:
display(
    mover_cohort.groupby("PRIMARY_PROCEDURE_NM")
    .count()["LOG_ID"]
    .reset_index()
    .sort_values("LOG_ID", ascending=False)
)

,PRIMARY_PROCEDURE_NM,LOG_ID
49,CRANIECTOMY,51
164,"FUSION, SPINE, LUMBAR AND LUMBOSACRAL, USING P...",41
234,"LAPAROTOMY, EXPLORATORY",35
229,"LAPAROSCOPY, DIAGNOSTIC",30
68,"CRANIOTOMY, WITH NEOPLASM EXCISION",28
...,...,...
202,IR ANGIO PULMONARY,1
203,IR CT BIOPSY LUNG,1
206,IR INSERT CHOLECYSTOSTOMY TUBE,1
207,IR KYPHOPLASTY THORACIC,1


In [16]:
pat_code = pd.read_csv(data_dir / "mover" / "patient_coding.csv")
pat_code["CATEGORY"] = pat_code.REF_BILL_CODE.apply(lambda x: str(x)[:2])
pat_code = pat_code.loc[~pat_code["CATEGORY"].isin(["02", "10"])]

In [17]:
rows = []
for k, grp in pd.merge(mover_cohort, pat_code, how="inner", on="MRN").groupby(
    ["LOG_ID"]
):
    rows.append(grp.sort_values("CATEGORY").iloc[[0]])

In [18]:
# VitalDb와 동일한 수준의 수술타입
pd.concat(rows).groupby("CATEGORY").count()["LOG_ID"].reset_index().sort_values(
    "LOG_ID", ascending=False
).head(20)

,CATEGORY,LOG_ID
0,00,404
2,03,140
6,07,96
3,04,91
10,0D,90
1,01,70
22,0T,39
19,0Q,31
9,0B,28
18,0P,24


In [19]:
pd.concat(rows).groupby("CATEGORY").count()["LOG_ID"].reset_index().sort_values(
    "LOG_ID", ascending=False
).sum()

CATEGORY    000307040D010T0Q0B0P0S05060R0H0J0F0G360K470908...
LOG_ID                                                   1199
dtype: object

# ASA

In [20]:
print(f"""
VitalDB ASA
* 1: {sum(vitaldb_cohort.asa == 1)} ({sum(vitaldb_cohort.asa == 1)/len(vitaldb_cohort):.2f})
* 2: {sum(vitaldb_cohort.asa == 2)} ({sum(vitaldb_cohort.asa == 2)/len(vitaldb_cohort):.2f})
* 3: {sum(vitaldb_cohort.asa == 3)} ({sum(vitaldb_cohort.asa == 3)/len(vitaldb_cohort):.2f})
* 4: {sum(vitaldb_cohort.asa == 4)} ({sum(vitaldb_cohort.asa == 4)/len(vitaldb_cohort):.2f})
* 5: {sum(vitaldb_cohort.asa == 5)} ({sum(vitaldb_cohort.asa == 5)/len(vitaldb_cohort):.2f})
* 6: {sum(vitaldb_cohort.asa == 6)} ({sum(vitaldb_cohort.asa == 6)/len(vitaldb_cohort):.2f})
* Missing: {sum(vitaldb_cohort.asa.isna())} ({sum(vitaldb_cohort.asa.isna())/len(vitaldb_cohort):.2f})
""")


VitalDB ASA
* 1: 624 (0.19)
* 2: 1994 (0.62)
* 3: 499 (0.15)
* 4: 29 (0.01)
* 5: 0 (0.00)
* 6: 12 (0.00)
* Missing: 81 (0.03)



In [21]:
print(f"""
MOVER ASA
* 1: {sum(mover_cohort.ASA_RATING_C == 1)} ({sum(mover_cohort.ASA_RATING_C == 1)/len(mover_cohort):.2f})
* 2: {sum(mover_cohort.ASA_RATING_C == 2)} ({sum(mover_cohort.ASA_RATING_C == 2)/len(mover_cohort):.2f})
* 3: {sum(mover_cohort.ASA_RATING_C == 3)} ({sum(mover_cohort.ASA_RATING_C == 3)/len(mover_cohort):.2f})
* 4: {sum(mover_cohort.ASA_RATING_C == 4)} ({sum(mover_cohort.ASA_RATING_C == 4)/len(mover_cohort):.2f})
* 5: {sum(mover_cohort.ASA_RATING_C == 5)} ({sum(mover_cohort.ASA_RATING_C == 5)/len(mover_cohort):.2f})
* 6: {sum(mover_cohort.ASA_RATING_C == 6)} ({sum(mover_cohort.ASA_RATING_C == 6)/len(mover_cohort):.2f})
* Missing: {sum(mover_cohort.ASA_RATING_C.isna())} ({sum(mover_cohort.ASA_RATING_C.isna())/len(mover_cohort):.2f})
""")


MOVER ASA
* 1: 11 (0.01)
* 2: 174 (0.15)
* 3: 801 (0.67)
* 4: 142 (0.12)
* 5: 16 (0.01)
* 6: 0 (0.00)
* Missing: 55 (0.05)



# Surgery duration

In [22]:
vsec = ((vitaldb_cohort.opend - vitaldb_cohort.opstart) % 60).astype(str)
vmin = (((vitaldb_cohort.opend - vitaldb_cohort.opstart) // 60) % 60).astype(str)
vhour = (((vitaldb_cohort.opend - vitaldb_cohort.opstart) // 60) // 60).astype(str)
vsd = (vhour + ":" + vmin + ":" + vsec).apply(lambda x: pd.Timedelta(x))
print(
    f"VitalDB: {vsd.median()} [{pd.Timedelta(np.quantile(vsd, .25))}, {pd.Timedelta(np.quantile(vsd, .75))}]"
)

VitalDB: 0 days 02:40:00 [0 days 01:41:46, 0 days 03:55:00]


In [23]:
mover_cohort.SURGERY_DATE

0        10/9/19 0:00
1        6/18/19 0:00
2        6/25/19 0:00
3        2/21/19 0:00
4        8/23/19 0:00
            ...      
1194    12/29/21 0:00
1195     1/20/23 0:00
1196      6/4/21 0:00
1197     3/22/22 0:00
1198     6/29/21 0:00
Name: SURGERY_DATE, Length: 1199, dtype: object

# ABP duration

In [24]:
config = load_config("../../config/conf.yaml")

In [25]:
src_path = Path(config["data_path"]).expanduser()
v_path = vitaldb_cohort.caseid.apply(
    lambda x: str(src_path / "vitaldb" / "01.raw" / str(x).zfill(4) / str(x).zfill(4))
    + "_100fs.h5"
).values
m_path = mover_cohort.PAT_ID.apply(
    lambda x: str(src_path / "mover" / "01.raw" / x / x) + "_100fs.h5"
).values

In [26]:
vabp_duration = []
for file_path in tqdm(v_path, ncols=75):
    with h5py.File(file_path, "r") as f:
        end = f["signal/ts"][-1].astype("datetime64[s]")
        start = f["signal/ts"][0].astype("datetime64[s]")
        duration = end - start
    vabp_duration.append(pd.Timedelta(duration))
vabp_duration = np.array(vabp_duration)
print(
    f"VitalDB: {np.median(vabp_duration)} [{pd.Timedelta(np.quantile(vabp_duration, .25))}, {pd.Timedelta(np.quantile(vabp_duration, .75))}]"
)

100%|██████████████████████████████████| 3239/3239 [02:05<00:00, 25.73it/s]

VitalDB: 0 days 03:41:23 [0 days 02:39:05, 0 days 04:59:38]


In [27]:
mabp_duration = []
for file_path in tqdm(m_path, ncols=75):
    with h5py.File(file_path, "r") as f:
        sig = f["signal"][:]
        t_offset = len(sig) // 100
        t_sec = str(t_offset % 60)
        t_min = str((t_offset // 60) % 60)
        t_hour = str((t_offset // 60) // 60)
    mabp_duration.append(f"{t_hour}:{t_min}:{t_sec}")
mabp_duration = pd.to_timedelta(np.array(mabp_duration))
print(
    f"MOVER: {pd.Timedelta(np.median(mabp_duration))} [{pd.Timedelta(np.quantile(mabp_duration, .25))}, {pd.Timedelta(np.quantile(mabp_duration, .75))}]"
)

100%|██████████████████████████████████| 1199/1199 [01:59<00:00, 10.00it/s]

MOVER: 0 days 06:59:32 [0 days 04:39:53.500000, 0 days 09:59:57]


# Anes duration

In [28]:
vsec = ((vitaldb_cohort.aneend - vitaldb_cohort.anestart) % 60).astype(int).astype(str)
vmin = (
    (((vitaldb_cohort.aneend - vitaldb_cohort.anestart) // 60) % 60)
    .astype(int)
    .astype(str)
)
vhour = (
    (((vitaldb_cohort.aneend - vitaldb_cohort.anestart) // 60) // 60)
    .astype(int)
    .astype(str)
)
vad = (vhour + ":" + vmin + ":" + vsec).apply(lambda x: pd.Timedelta(x))
print(
    f"VitalDB: {vad.median()} [{pd.Timedelta(np.quantile(vad, .25))}, {pd.Timedelta(np.quantile(vad, .75))}]"
)

VitalDB: 0 days 03:52:00 [0 days 02:50:00, 0 days 05:14:00]


In [29]:
mad = pd.to_datetime(mover_cohort.AN_STOP_DATETIME) - pd.to_datetime(
    mover_cohort.AN_START_DATETIME
)
print(
    f"MOVER: {mad.median()} [{pd.Timedelta(np.quantile(mad, .25))}, {pd.Timedelta(np.quantile(mad, .75))}]"
)

MOVER: 0 days 04:33:00 [0 days 03:02:30, 0 days 06:45:30]


/var/folders/2g/clmvrww15hzf0cp5zjmw2x0m0000gn/T/ipykernel_85300/1266268479.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  mad = pd.to_datetime(mover_cohort.AN_STOP_DATETIME) - pd.to_datetime(
/var/folders/2g/clmvrww15hzf0cp5zjmw2x0m0000gn/T/ipykernel_85300/1266268479.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  mad = pd.to_datetime(mover_cohort.AN_STOP_DATETIME) - pd.to_datetime(


# Waveform missingness

In [30]:
config = load_config("../../config/conf.yaml")

In [31]:
src_path = Path(config["data_path"]).expanduser()
v_path = vitaldb_cohort.caseid.apply(
    lambda x: str(src_path / "vitaldb" / "01.raw" / str(x).zfill(4) / str(x).zfill(4))
    + "_100fs.h5"
).values
m_path = mover_cohort.PAT_ID.apply(
    lambda x: str(src_path / "mover" / "01.raw" / x / x) + "_100fs.h5"
).values

In [32]:
vmiss = []
for file_path in tqdm(v_path, ncols=75):
    with h5py.File(file_path, "r") as f:
        sig = f["signal/abp"][:].reshape(-1)
    vmiss.append(np.isnan(sig).sum() / len(sig))
vmiss = np.array(vmiss)
print(
    f"VitalDB: {np.nanmedian(vmiss)} [{np.quantile(vmiss, 0.25)}, {np.quantile(vmiss, 0.75)}]"
)
print("<0.01%")

100%|██████████████████████████████████| 3239/3239 [01:42<00:00, 31.51it/s]

VitalDB: 8.654555944916297e-05 [0.0, 0.0013958110220039126]
<0.01%


In [33]:
mmiss = []
for file_path in tqdm(m_path, ncols=75):
    with h5py.File(file_path, "r") as f:
        sig = f["signal"][:].reshape(-1)
    mmiss.append(np.isnan(sig).sum() / len(sig))
mmiss = np.array(mmiss)
print(
    f"MOVER: {np.nanmedian(mmiss)} [{np.quantile(mmiss, 0.25)}, {np.quantile(mmiss, 0.75)}]"
)

100%|██████████████████████████████████| 1199/1199 [01:28<00:00, 13.54it/s]

MOVER: 0.0 [0.0, 0.0]


# Hypotension prevalence

In [34]:
config = load_config("../../config/conf.yaml")

In [35]:
src_path = Path(config["data_path"]).expanduser()

In [36]:
v_label = sorted(
    filter(
        lambda x: (x.name[0] != ".") and (x.name.split("_")[-1] != "segments.npy"),
        (src_path / "vitaldb" / "02_04.processed").rglob(
            "*hypophetversion2*actual*.npy"
        ),
    )
)
v_label = np.array(v_label)
v_events = [load_npy(x) for x in v_label]
is_v_hypo = [True if len(x) != 0 else False for x in v_events]
v_hypo_cid = np.array([x.parent.name for x in v_label])[is_v_hypo].astype(int)
print(
    f"VitalDB: {len(vitaldb_cohort.query('caseid.isin(@v_hypo_cid)'))} cases ({len(vitaldb_cohort.query('caseid.isin(@v_hypo_cid)')) / len(vitaldb_cohort)})"
)

VitalDB: 2205 cases (0.6807656684161778)


In [37]:
m_label = sorted(
    filter(
        lambda x: (x.name[0] != "."),
        (src_path / "mover" / "03.label").rglob("*hypophetversion2*actual*.npy"),
    )
)
m_label = np.array(m_label)
m_events = [load_npy(x) for x in m_label]
is_m_hypo = [True if len(x) != 0 else False for x in m_events]
m_hypo_cid = np.array([x.parent.name for x in m_label])[is_m_hypo]
print(
    f"MOVER: {len(mover_cohort.query('PAT_ID.isin(@m_hypo_cid)'))} cases ({len(mover_cohort.query('PAT_ID.isin(@m_hypo_cid)')) / len(mover_cohort)})"
)

MOVER: 694 cases (0.5788156797331109)


# Incidence of hypotension

In [38]:
config = load_config("../../config/conf.yaml")
src_path = Path(config["data_path"]).expanduser()

In [39]:
v_label = sorted(
    filter(
        lambda x: (x.name[0] != ".") and (x.name.split("_")[-1] != "segments.npy"),
        (src_path / "vitaldb" / "02_04.processed").rglob(
            "*hypophetversion2*actual*.npy"
        ),
    )
)
v_label = np.array(v_label)
v_events = [load_npy(x) for x in v_label]
is_v_hypo = np.array([len(x) for x in v_events])
is_v_hypo2 = [len(x) for x in filter(lambda x: len(x) != 0, v_events)]
print(
    f"VitalDB: {np.median(is_v_hypo)} [{np.quantile(is_v_hypo, 0.25)}, {np.quantile(is_v_hypo, 0.75)}]"
)
print(
    f"VitalDB (at least 1 pos): {np.median(is_v_hypo2)} [{np.quantile(is_v_hypo2, 0.25)}, {np.quantile(is_v_hypo2, 0.75)}]"
)

VitalDB: 2.0 [0.0, 6.0]
VitalDB (at least 1 pos): 4.0 [2.0, 9.0]


In [40]:
m_label = sorted(
    filter(
        lambda x: (x.name[0] != "."),
        (src_path / "mover" / "03.label").rglob("*hypophetversion2*actual*.npy"),
    )
)
m_label = np.array(m_label)
m_events = [load_npy(x) for x in m_label]
is_m_hypo = np.array([len(x) for x in m_events])
is_m_hypo2 = [len(x) for x in filter(lambda x: len(x) != 0, m_events)]
print(
    f"MOVER: {np.median(is_m_hypo)} [{np.quantile(is_m_hypo, 0.25)}, {np.quantile(is_m_hypo, 0.75)}]"
)
print(
    f"MOVER (at least 1 pos): {np.median(is_m_hypo2)} [{np.quantile(is_m_hypo2, 0.25)}, {np.quantile(is_m_hypo2, 0.75)}]"
)

MOVER: 0.0 [0.0, 1.0]
MOVER (at least 1 pos): 3.0 [1.0, 6.0]


# Baseline blood pressure (MAP)

SQI까지 전부 거친 신호들을 대상으로 추출.

In [41]:
config = load_config("../../config/conf.yaml")
src_path = Path(config["data_path"]).expanduser()

In [42]:
vdb_pos_seg = np.hstack(
    [
        load_npy(fp)
        for fp in (src_path / "vitaldb" / "05.dataset_new").rglob(
            "hypophetversion2*pos_segments.npy"
        )
    ]
)
vdb_neg_seg = np.hstack(
    [
        load_npy(fp)
        for fp in (src_path / "vitaldb" / "05.dataset_new").rglob(
            "hypophetversion2*neg_segments.npy"
        )
    ]
)
vdb_seg = np.vstack([vdb_pos_seg, vdb_neg_seg])
print(vdb_pos_seg.shape, vdb_neg_seg.shape, vdb_seg.shape)

(79028, 3000) (147951, 3000) (226979, 3000)


In [43]:
vdb_map = np.mean(vdb_seg, axis=1)
print(
    f"Mean arterial pressure: {np.percentile(vdb_map, 50)} [{np.percentile(vdb_map, 25)}, {np.percentile(vdb_map, 75)}]"
)

Mean arterial pressure: 82.87067413330078 [74.7292251586914, 92.72396087646484]


In [44]:
mov_pos_seg = np.hstack(
    [
        load_npy(fp)
        for fp in (src_path / "mover" / "05.dataset_new").rglob(
            "hypophetversion2*pos_segments.npy"
        )
    ]
)
mov_neg_seg = np.hstack(
    [
        load_npy(fp)
        for fp in (src_path / "mover" / "05.dataset_new").rglob(
            "hypophetversion2*neg_segments.npy"
        )
    ]
)
mov_seg = np.vstack([mov_pos_seg, mov_neg_seg])
print(mov_pos_seg.shape, mov_neg_seg.shape, mov_seg.shape)

(9295, 3000) (16498, 3000) (25793, 3000)


In [45]:
mov_map = np.mean(mov_seg, axis=1)
print(
    f"Mean arterial pressure: {np.percentile(mov_map, 50)} [{np.percentile(mov_map, 25)}, {np.percentile(mov_map, 75)}]"
)

Mean arterial pressure: 81.22657012939453 [74.05564880371094, 90.28704071044922]


# Segments

In [46]:
config = load_config("../../config/conf.yaml")
src_path = Path(config["data_path"]).expanduser()

In [47]:
vdb_pos_cid = np.hstack(
    [
        load_npy(fp)
        for fp in (src_path / "vitaldb" / "05.dataset_new").rglob(
            "hypophetversion2*pos_cid.npy"
        )
    ]
)
vdb_neg_cid = np.hstack(
    [
        load_npy(fp)
        for fp in (src_path / "vitaldb" / "05.dataset_new").rglob(
            "hypophetversion2*neg_cid.npy"
        )
    ]
)
vdb_cid = np.hstack([vdb_pos_cid, vdb_neg_cid])
print(vdb_pos_cid.shape, vdb_neg_cid.shape, vdb_cid.shape)
print(f"patient no: {np.unique(vdb_cid).size}")

(79028,) (147951,) (226979,)
patient no: 3239


In [48]:
k, v = np.unique(vdb_cid, return_counts=True)
print(
    f"Segments per patient: {np.percentile(v, 50)} [{np.percentile(v, 25)}, {np.percentile(v, 75)}]"
)

Segments per patient: 63.0 [38.0, 97.0]


In [49]:
mov_pos_cid = np.hstack(
    [
        load_npy(fp)
        for fp in (src_path / "mover" / "05.dataset_new").rglob(
            "hypophetversion2*pos_cid.npy"
        )
    ]
)
mov_neg_cid = np.hstack(
    [
        load_npy(fp)
        for fp in (src_path / "mover" / "05.dataset_new").rglob(
            "hypophetversion2*neg_cid.npy"
        )
    ]
)
mov_cid = np.hstack([mov_pos_cid, mov_neg_cid])
print(mov_pos_cid.shape, mov_neg_cid.shape, mov_cid.shape)
print(f"patient no: {np.unique(mov_cid).size}")

(9295,) (16498,) (25793,)
patient no: 649


In [50]:
k, v = np.unique(mov_cid, return_counts=True)
print(
    f"Segments per patient: {np.percentile(v, 50)} [{np.percentile(v, 25)}, {np.percentile(v, 75)}]"
)

Segments per patient: 34.0 [11.0, 57.0]
